# Demo 2 — GoalScope: verbalizing the next sub-goal before it is written

**Question.** The model is in the middle of a function that has to compute several things in order. It has finished
some of them and is about to write the next one. *Before* it writes a single token of that next line, does it already
know **which sub-task** comes next? And can we read it out?

**Paper.** §3.2–3.3 (`Text/03_what.tex`, `fig:goalscope`) and App. "GoalScope" (`Text/Appendix/goalscope.tex`).

**Method in one line.** Take the summed write of the 122 **goal heads** (found in Demo 1) at a single source token and
add it to the residual stream of a *different* prompt, a probe that asks *"What are you doing at this line of the
code?"*. The model answers with whatever the injected vector encodes.

**What you'll see.**
1. One multi-sub-task prompt from the locked evaluation corpus, prefilled up to the next sub-task (the *frontier*).
2. GoalScope read-outs at the four **pivot tokens** right before the frontier line (`prev_end`, `indent`, `var`, `eq`),
   in a table and in the paper-style viewer.
3. **Causality probe**: the same vector injected into an empty C++ stub `bool check(auto value){`. If the vector carries
   the sub-goal, the model *implements* it, in another language.
4. A **residual-stream baseline** at the same tokens, for contrast.
5. A cross-check against the cached paper run on this exact example.

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file, so you can read
it without a GPU.

**The setup** (schematics from the lab talk):

![GoalScope: sum the goal heads' outputs at one token and inject them into a neutral probe (§3.2, App. D).](figures/02_goalscope_method.png)

*GoalScope: sum the goal heads' outputs at one token and inject them into a neutral probe (§3.2, App. D).*

![The two evaluations: faithfulness (the model verbalizes the sub-goal) and causality (an empty C++ stub implements it).](figures/02_goalscope_evaluation.png)

*The two evaluations: faithfulness (the model verbalizes the sub-goal) and causality (an empty C++ stub implements it).*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
huggingface_hub.utils.disable_progress_bars()

repo root: /disk/u/arnab/Codes/Projects/goals


## Config
The few knobs. The defaults are the paper's GoalScope configuration (`scripts/evaluation/eval_goalscope.py` argparse
defaults).

**Why this example (`list[int]_python_016`).** It has four sub-tasks and three of them are already written, so the
frontier is the *last* one ("whether any value appears more than once"). Nothing in the code so far mentions duplicates;
the only place the sub-goal appears is the numbered list in the instruction. In the paper run, this spec got
oracle-graded **YES** at all four pivots under both faithfulness and causality. We picked it from the 60 such specs
(out of 253) because the frontier sub-task is easy to recognise in either language.

In [2]:
CUDA        = '7'                                   # one free 80 GB GPU
MODEL       = 'Qwen/Qwen2.5-Coder-32B-Instruct'
HEAD_SET    = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal__lam20_thr0.9_band25-56.json'   # the 122 goal heads
SPEC_ID     = 'list[int]_python_016'                 # example from data/goalscope_eval.json
STRATEGY    = 'additive_norm'                        # h <- h + alpha * ||h|| * g/||g||
ALPHA       = 0.1
STEER_LAYERS = (35, 46)                              # half-open: layers 35..45 receive the injection
SEED        = 123
RUN_JUDGE   = True                                   # Claude judge (cached on disk; paper rubric)
RESID_LAYERS = (40, 50)                              # residual baseline: mean residual over layers 40..49

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model and the goal heads
The head set is a plain `[[layer, head], ...]` json. Every reported GoalScope run reads it.

In [3]:
import json, torch
from rich import print as rprint
from rich.console import Console
from rich.table import Table
from rich.markup import escape
from IPython.display import HTML, display
import html as _html

from src.models import ModelandTokenizer
from src.utils import experiment_utils
from src.goals.head_sets import load_json_heads
from src.goals.goalscope import apply_goalscope, _build_goalscope_prompt, _build_causality_prompt, CAUSAL_PREFIX
from src.goals.goalscope_viz import goalscope_qualitative_html
from scripts.evaluation.eval_goalscope import (
    DEFAULT_DATASET, ALL_DATATYPES, MAX_NEW_TOKENS, GEN_MAX_NEW_TOKENS,
    load_locked_datapoints, build_prompt, generate_completion, frontier_impl_span,
    pivot_positions, hard_eval,
)
from scripts.evaluation.eval_readout_sources import residual_patchscope

console = Console(width=130, highlight=False)
torch.set_grad_enabled(False)
experiment_utils.set_seed(SEED)

mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map={'': 0}, attn_implementation='eager')
heads = load_json_heads(HEAD_SET, mt.n_layer, mt.n_heads)
print(f'{MODEL}: {mt.n_layer} layers x {mt.n_heads} heads | goal heads: {len(heads)} '
      f'(layers {min(l for l, _ in heads)}-{max(l for l, _ in heads)})')

Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct: 64 layers x 40 heads | goal heads: 122 (layers 28-55)


## 1. The task, prefilled to the frontier
`data/goalscope_eval.json` is the locked corpus: 253 multi-sub-task specs over four input types. Each spec lists 2–4
sub-tasks, prefills the code for the first *j* of them, and stops at `{var} =` of the next one. `build_prompt` is the
exact prompt builder used by the evaluation.

In [4]:
datapoints, meta = load_locked_datapoints(DEFAULT_DATASET, ALL_DATATYPES)
inst = dict(datapoints)[SPEC_ID]
prompt, var = build_prompt(mt, inst)

t = Table(title=f'{escape(SPEC_ID)}: sub-tasks in order', show_lines=False)
t.add_column('#'); t.add_column('sub-task id'); t.add_column('status')
done = set(inst.src_progress)
for k, p in enumerate(inst.task.predicates, 1):
    st = 'written (prefilled)' if p.id in done else ('[bold magenta]FRONTIER: next to write[/]' if p.id == inst.expected_next else 'later')
    t.add_row(str(k), p.id, st)
console.print(t)
print(f'frontier = {inst.expected_next!r}: "{inst.expected_next_desc}"  -> variable `{var}`\n')
print('--- prompt (chat template, assistant turn prefilled) ---')
print(prompt)

    list[int]_python_016: sub-tasks in order    
┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ sub-task id    ┃ status                  ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ count_odd      │ written (prefilled)     │
│ 2 │ count_positive │ written (prefilled)     │
│ 3 │ is_sorted      │ written (prefilled)     │
│ 4 │ has_duplicate  │ FRONTIER: next to write │
└───┴────────────────┴─────────────────────────┘

frontier = 'has_duplicate': "whether any value appears more than once"  -> variable `d`

--- prompt (chat template, assistant turn prefilled) ---
<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `func` which, given a list of integers `nums`, computes in sequential order:
(1) the number of odd values
(2) the number of positive values
(3) whether the values are sorted in non-decreasing order
(4) whether any value appears more than once
<|im_end|>
<|im_start|>assistant
<code>
def func(nums: list[int]):
    a = sum(1 for x in nums if x % 2 != 0)
    b = sum(1 for x in nums if x > 0)
    c = all(nums[i] <= nums[i+1] for i in range(len(nums)-1))
    d =


## 2. Let the model write the frontier line
Greedy decoding (64 tokens). The first generated line, together with `{var} =`, is the **implementation span** that
the read-outs are graded against. The read-outs below come from tokens that are all *before* this span.

In [5]:
completion = generate_completion(mt, prompt, GEN_MAX_NEW_TOKENS)
full_text = prompt + completion
impl_span = frontier_impl_span(prompt, completion, var)
rprint('[dim]' + escape(prompt.split('<code>')[-1]) + '[/dim]' + '[bold green]' + escape(completion) + '[/bold green]')
print('\nimplementation span (gold):', impl_span)

def func(nums: list[int]):
    a = sum(1 for x in nums if x % 2 != 0)
    b = sum(1 for x in nums if x > 0)
    c = all(nums[i] <= nums[i+1] for i in range(len(nums)-1))
    d = len(nums) != len(set(nums))
    return a, b, c, d
</code>


implementation span (gold): d = len(nums) != len(set(nums))


## 3. The pivot tokens
The four last tokens of the prompt: the newline ending the previous line (`prev_end`), the indentation (`indent`), the
variable name (`var`) and `=` (`eq`). None of them carries any lexical information about *which* sub-task comes next.
That information can only come from the model's internal plan.

In [6]:
pivs = pivot_positions(mt, prompt, full_text, ['prev_end', 'indent', 'var', 'eq'])
positions = sorted(pivs.values())
from src.tokens import prepare_input
ids = prepare_input(prompts=full_text, tokenizer=mt)['input_ids'][0].tolist()
t = Table(title='pivots (absolute token positions in full_text)')
t.add_column('pivot'); t.add_column('pos', justify='right'); t.add_column('token')
for a, p in pivs.items():
    t.add_row(a, str(p), repr(mt.tokenizer.decode([ids[p]])))
console.print(t)

  pivots (absolute token   
  positions in full_text)  
┏━━━━━━━━━━┳━━━━━┳━━━━━━━━┓
┃ pivot    ┃ pos ┃ token  ┃
┡━━━━━━━━━━╇━━━━━╇━━━━━━━━┩
│ prev_end │ 180 │ '))\n' │
│ indent   │ 181 │ '   '  │
│ var      │ 182 │ ' d'   │
│ eq       │ 183 │ ' ='   │
└──────────┴─────┴────────┘

## 4. GoalScope read-outs
For each pivot token, `apply_goalscope`:
1. runs the source prompt once and sums the 122 goal heads' o_proj writes at that token into one vector $g$
   (`collect_head_steering_vecs`);
2. runs the probe *"What are you doing at this line of the code. Keep your answer as brief as possible."* and, at
   every layer in 35–45 and at every decode step, adds $\alpha\,\lVert h\rVert\, g/\lVert g\rVert$ to the last
   token's residual $h$ (`additive_norm`, $\alpha=0.1$);
3. decodes greedily for 64 tokens.

The probe itself says nothing about the code. Whatever the model talks about comes from $g$.

In [7]:
gs_kwargs = dict(selected_heads=heads, strategy=STRATEGY, alpha=ALPHA, steer_layer_range=STEER_LAYERS,
                 n_steer_tokens='all', max_new_tokens=MAX_NEW_TOKENS)
res = apply_goalscope(full_text, mt=mt, token_positions=positions, include_noncode=True, **gs_kwargs)
faith = {r['pos']: r['patched_gen'] for r in res if r['scoped']}

t = Table(title=f'GoalScope read-outs (frontier: "{inst.expected_next_desc}")', show_lines=True)
t.add_column('pivot'); t.add_column('token'); t.add_column('read-out', max_width=100)
for a, p in pivs.items():
    t.add_row(a, repr(mt.tokenizer.decode([ids[p]])), escape(faith[p].strip()))
console.print(t)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


                         GoalScope read-outs (frontier: "whether any value appears more than once")                         
┏━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pivot    ┃ token  ┃ read-out                                                                                             ┃
┡━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ prev_end │ '))\n' │ I'm checking if the current index is the last index or if there's a duplicate by comparing it to the │
│          │        │ set of the list.                                                                                     │
├──────────┼────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ indent   │ '   '  │ I'm checking for duplicates by comparing the length of the list to the length of the set created     │
│          │        │ from the list.                                                                                       │
├──────────┼────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ var      │ ' d'   │ I'm checking for duplicates by converting the list to a set and comparing its length to the original │
│          │        │ list's length.                                                                                       │
├──────────┼────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ eq       │ ' ='   │ I'm checking if there are any duplicate elements in the list by comparing the length of the list     │
│          │        │ with the length of the set created from the list. If they differ, duplicates exist.                  │
└──────────┴────────┴──────────────────────────────────────────────────────────────────────────────────────────────────────┘

### Paper-style view
`src.goals.goalscope_viz.goalscope_qualitative_html` is the renderer used for the paper's qualitative figures and the
companion site. The four pivot read-outs are pinned. Hover or click a highlighted token to inspect it. The page is
also saved as a standalone `.html` next to this notebook. (The viewer needs JavaScript. In a static renderer such as
GitHub, use the table above.)

In [8]:
# show the instruction + code (drop the system-prompt tokens), pin the four pivots
start_char = full_text.index('Write a Python')
offs = prepare_input(prompts=full_text, tokenizer=mt, return_offsets_mapping=True)['offset_mapping'][0].tolist()
start_tok = next(i for i, (s, e) in enumerate(offs) if e > start_char)
rows = [r for r in res if r['pos'] >= start_tok]
os.makedirs('notebooks/demos/outputs', exist_ok=True)
doc = goalscope_qualitative_html(rows, title=f'GoalScope · {SPEC_ID} · 122 goal heads',
                                 save_path='notebooks/demos/outputs/02_goalscope_viewer.html',
                                 initial_pins=positions)
display(HTML(f'<iframe srcdoc="{_html.escape(doc, quote=True)}" style="width:100%;height:560px;'
             f'border:1px solid #e5e7eb;border-radius:10px;" sandbox="allow-scripts"></iframe>'))

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## 5. Faithfulness judge
The paper grades every read-out with Claude (`claude-sonnet-4-6`) using the rubric `FAITH_HARD_SYS` in
`eval_goalscope.py`: does the description, past its surface noise, describe the *same sub-task* as the implementation
span? `hard_eval` is the exact grading function. Its calls are cached on disk, so a reproduced read-out costs no API
call.

In [9]:
def judge_table(readouts, mode, title):
    t = Table(title=title, show_lines=True)
    t.add_column('pivot'); t.add_column('verdict'); t.add_column('read-out', max_width=95)
    verdicts = {}
    for a, p in pivs.items():
        ro = readouts[p]
        v = hard_eval(ro, impl_span, mode)['verdict'] if RUN_JUDGE else '-'
        verdicts[a] = v
        col = 'green' if v == 'YES' else 'red'
        t.add_row(a, f'[{col}]{v}[/]', escape(ro.strip()))
    console.print(t)
    return verdicts

print('gold implementation span:', impl_span)
faith_verdicts = judge_table(faith, 'faithfulness', 'faithfulness (goal heads)')

gold implementation span: d = len(nums) != len(set(nums))


                                               faithfulness (goal heads)                                                
┏━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pivot    ┃ verdict ┃ read-out                                                                                        ┃
┡━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ prev_end │ YES     │ I'm checking if the current index is the last index or if there's a duplicate by comparing it   │
│          │         │ to the set of the list.                                                                         │
├──────────┼─────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ indent   │ YES     │ I'm checking for duplicates by comparing the length of the list to the length of the set        │
│          │         │ created from the list.                                                                          │
├──────────┼─────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ var      │ YES     │ I'm checking for duplicates by converting the list to a set and comparing its length to the     │
│          │         │ original list's length.                                                                         │
├──────────┼─────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ eq       │ YES     │ I'm checking if there are any duplicate elements in the list by comparing the length of the     │
│          │         │ list with the length of the set created from the list. If they differ, duplicates exist.        │
└──────────┴─────────┴─────────────────────────────────────────────────────────────────────────────────────────────────┘

## 6. Causality probe: the vector *implements* the sub-goal in C++
A faithful description could in principle be produced by a vector that merely *correlates* with the sub-goal. The
causality probe asks for more. Inject the same vector into an information-less C++ stub,

```
system: You are a helpful assistant proficient in programming.
user:   You are implementing a simple C++ function that checks for a condition. Just write the function body ...
assistant: bool check(auto value){
```

The vector passes if the model writes the sub-goal's logic. First, the **unsteered** stub, to see what the model writes
with no goal signal (the judge's sanitizer treats this sign/zero default as an automatic NO):

In [10]:
causal_prompt = _build_causality_prompt(mt)
default = generate_completion(mt, causal_prompt, MAX_NEW_TOKENS)
print('unsteered stub ->', CAUSAL_PREFIX + default)

unsteered stub -> bool check(auto value){ return value > 0; }


In [11]:
res_c = apply_goalscope(full_text, mt=mt, token_positions=positions, goalscope_prompt=causal_prompt, **gs_kwargs)
causal = {r['pos']: CAUSAL_PREFIX + r['patched_gen'] for r in res_c}
print('gold implementation span (Python):', impl_span)
causal_verdicts = judge_table(causal, 'causality', 'causality (goal heads -> empty C++ stub)')

gold implementation span (Python): d = len(nums) != len(set(nums))


                                        causality (goal heads -> empty C++ stub)                                        
┏━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pivot    ┃ verdict ┃ read-out                                                                                        ┃
┡━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ prev_end │ YES     │ bool check(auto value){ return std::find_if(value.begin(), value.end(), [](auto it){return it   │
│          │         │ == *std::next(it);}) != value.end() && std::set(value.begin(), value.end()).size() ==           │
│          │         │ value.size(); }                                                                                 │
├──────────┼─────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ indent   │ YES     │ bool check(auto value){ return std::set(value.begin(), value.end()).size() != value.size(); }   │
├──────────┼─────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ var      │ YES     │ bool check(auto value){ return std::find_if(value.begin(), value.end(), [](auto const& v){      │
│          │         │ return std::unordered_set<decltype(v)>(value.begin(), value.end()).size() != value.size(); });  │
│          │         │ }                                                                                               │
├──────────┼─────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ eq       │ YES     │ bool check(auto value){ return std::set(value.begin(), value.end()).size() != value.size(); }   │
└──────────┴─────────┴─────────────────────────────────────────────────────────────────────────────────────────────────┘

## 7. Baseline: the residual stream at the same token
Is this just "any hidden state at this token works"? `eval_readout_sources.residual_patchscope` runs the same probe,
the same steer band and the same $\alpha$, but injects the residual stream (mean over layers 40–49) instead of the goal
heads' write. The residual stream holds everything about the current token (its identity, its position, the
surrounding syntax), and that swamps the sub-goal. In the paper this baseline scores about 1%.

In [12]:
src_layers = list(range(*RESID_LAYERS))
r_f = residual_patchscope(mt, full_text, positions, src_layers, alpha=ALPHA, source='resid')
r_c = residual_patchscope(mt, full_text, positions, src_layers, alpha=ALPHA, source='resid', probe_prompt=causal_prompt)
resid_faith = {r['pos']: r['patched_gen'] for r in r_f}
resid_causal = {r['pos']: CAUSAL_PREFIX + r['patched_gen'] for r in r_c}
rf_verdicts = judge_table(resid_faith, 'faithfulness', f'faithfulness (residual L{RESID_LAYERS[0]}-{RESID_LAYERS[1]-1})')
rc_verdicts = judge_table(resid_causal, 'causality', f'causality (residual L{RESID_LAYERS[0]}-{RESID_LAYERS[1]-1})')

                                              faithfulness (residual L40-49)                                               
┏━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pivot    ┃ verdict    ┃ read-out                                                                                        ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ prev_end │ NO_INFO    │ I'm checking the                                                                                │
├──────────┼────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ indent   │ YES        │ I return unique elements return set return len unique_elements return len unique_elements       │
│          │            │ return return unique return return return return return return return return return return      │
│          │            │ return return return return return return return return return return return return return      │
│          │            │ return return return return return return return return return return return return return      │
│          │            │ return return return return return return return return return return return                    │
├──────────┼────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ var      │ HEAVY_LOOP │ Executing = = ( len > e = = ( len > = = = = = = = = = = d = = e = = e = = d = = e = = d = = e = │
│          │            │ d = = e = d = d = d = e = d = d = d = d = d                                                     │
├──────────┼────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ eq       │ HEAVY_LOOP │ I len len len len len len len len len len len len len len len len len len len len len len len   │
│          │            │ len len len len len len len len len len len len len len len len len len len len len len len len │
│          │            │ len len len len len len len len len len len len len len len len                                 │
└──────────┴────────────┴─────────────────────────────────────────────────────────────────────────────────────────────────┘

                                                causality (residual L40-49)                                                
┏━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pivot    ┃ verdict    ┃ read-out                                                                                        ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ prev_end │ NO         │ bool check(auto value){    return    return    return    return    return    return    return   │
│          │            │ return    return    return    return    return    return    return    return    return          │
│          │            │ return    return    return    return    return    return    return    return    return          │
│          │            │ return    return    return    return    return    return    return                              │
├──────────┼────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ indent   │ HEAVY_LOOP │ bool check(auto value){ return return std:: return unique return std return return return       │
│          │            │ return return return return return return return return return return return return return      │
│          │            │ return return return return return return return return return return return return return      │
│          │            │ return return return return return return return return return return return return return      │
│          │            │ return return return return return return return return return return return return return      │
│          │            │ return                                                                                          │
├──────────┼────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ var      │ HEAVY_LOOP │ bool check(auto value){ = ( value >  = ( value >  = ( value > d = ( e = ( e = e = ( e = e = e = │
│          │            │ e = e = e = e = e = d = e = e = e = e = e = d = e = e = d = e = e                               │
├──────────┼────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
│ eq       │ HEAVY_LOOP │ bool check(auto value){ std value std value len len len len len len len len len len len len len │
│          │            │ len len len len len len len len len len len len len len len len len len len len len len len len │
│          │            │ len len len len len len len len len len len len len len len len len len len len len len len     │
└──────────┴────────────┴─────────────────────────────────────────────────────────────────────────────────────────────────┘

In [13]:
t = Table(title=f'{escape(SPEC_ID)}: oracle verdicts per pivot')
t.add_column('read-out source'); t.add_column('probe')
for a in pivs: t.add_column(a)
for name, probe, v in [('122 goal heads', 'faithfulness', faith_verdicts), ('122 goal heads', 'causality', causal_verdicts),
                       ('residual L40-49', 'faithfulness', rf_verdicts), ('residual L40-49', 'causality', rc_verdicts)]:
    t.add_row(name, probe, *[('[green]' if v[a] == 'YES' else '[red]') + v[a] + '[/]' for a in pivs])
console.print(t)

                  list[int]_python_016: oracle verdicts per pivot                   
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ read-out source ┃ probe        ┃ prev_end ┃ indent     ┃ var        ┃ eq         ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 122 goal heads  │ faithfulness │ YES      │ YES        │ YES        │ YES        │
│ 122 goal heads  │ causality    │ YES      │ YES        │ YES        │ YES        │
│ residual L40-49 │ faithfulness │ NO_INFO  │ YES        │ HEAVY_LOOP │ HEAVY_LOOP │
│ residual L40-49 │ causality    │ NO       │ HEAVY_LOOP │ HEAVY_LOOP │ HEAVY_LOOP │
└─────────────────┴──────────────┴──────────┴────────────┴────────────┴────────────┘

With the goal heads, every pivot gives a clean description (or C++ implementation) of the duplicate check. With the
residual stream, most read-outs collapse into repetition loops. The judge labels these `HEAVY_LOOP` or `NO_INFO`.
Watch out for the occasional "YES" on a looping residual read-out: a fragment such as "unique elements ... set ... len"
can be enough for the judge. Over the whole corpus the residual baseline stays at about 1%. Single examples are
illustrations, not scores.

## 8. Cross-check against the cached paper run
GoalScope is deterministic (greedy, the same batch of four positions as the evaluation), so the read-outs above should
match the reported run's `records.jsonl` for this spec. (Exact string equality can break under bf16 kernel
nondeterminism on a different GPU or driver. The verdicts are what count.)

In [14]:
from src.utils import env_utils
leaf = 'heads=goal__layers=35-46__nsteer=all__maxnew=64__span=frontier__dtype=all__nspec=253__nc=2-3-4__seed=123'
base = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'evaluation', 'goalscope', MODEL.split('/')[-1])
t = Table(title='this notebook vs. the reported run', show_lines=False)
t.add_column('mode'); t.add_column('pivot'); t.add_column('paper verdict'); t.add_column('read-out identical?')
for mode, mine in [('faithfulness', faith), ('causality', causal)]:
    path = os.path.join(base, mode, 'additive_norm', '0.1', leaf, 'records.jsonl')
    if not os.path.exists(path):
        print('not found:', path); continue
    recs = {r['anchor']: r for r in map(json.loads, open(path)) if r['spec'] == SPEC_ID}
    for a, p in pivs.items():
        same = recs[a]['readout'].strip() == mine[p].strip()
        t.add_row(mode, a, recs[a]['hard'], str(same))
console.print(t)

               this notebook vs. the reported run                
┏━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ mode         ┃ pivot    ┃ paper verdict ┃ read-out identical? ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ faithfulness │ prev_end │ YES           │ True                │
│ faithfulness │ indent   │ YES           │ True                │
│ faithfulness │ var      │ YES           │ True                │
│ faithfulness │ eq       │ YES           │ True                │
│ causality    │ prev_end │ YES           │ True                │
│ causality    │ indent   │ YES           │ True                │
│ causality    │ var      │ YES           │ True                │
│ causality    │ eq       │ YES           │ True                │
└──────────────┴──────────┴───────────────┴─────────────────────┘

## Where this goes in the paper / how to reproduce at scale
- **Full evaluation** (253 specs × 5 pivots, Claude judge):
  ```
  python -m scripts.evaluation.eval_goalscope --mode faithfulness   # defaults = the paper config used here
  python -m scripts.evaluation.eval_goalscope --mode causality
  ```
  Job list: `job_list/goalscope_eval/goalscope_eval_a_residual.json` (jobs `goalscope_a_goalheads_{faithfulness,causality}`).
  Results: `results/evaluation/goalscope/Qwen2.5-Coder-32B-Instruct/<mode>/additive_norm/0.1/heads=goal__.../summary.json`.
  Smoke run: `--limit 4`.
- **Paper numbers** (oracle YES rate per pivot `prev_end / indent / var / eq`, Qwen2.5-Coder-32B-Instruct, 122 goal heads):
  faithfulness 58.9 / 94.5 / 90.5 / 97.2 % (mean 85.3 %); causality 52.2 / 80.2 / 71.9 / 95.7 % (mean 75.0 %).
  `prev_end` is the hardest: there the model has only just finished the previous line.
- **Baselines** (App. "Goal Heads vs. Other Readout Sources", `scripts/evaluation/eval_readout_sources.py`): attention
  output L40–50 32.6 / 29.2 %, residual L40–50 1.2 / 0.6 %, residual L25–56 3.5 / 2.0 % (mean faithfulness / causality).
- **Figure**: bar panel of `fig:goalscope` = `notebooks/fig_n_table_generators/readout_sources_and_head_budget.ipynb`.
- Where the goal heads come from: **Demo 1** (`01_goal_head_localization.ipynb`).